# 08 — Governance: owners, tiers, tags, glossary, domains

**What you will learn**
- Ownership: teams and users, owners on assets — and why owners turn alerts and incidents into actions.
- Tiers (`Tier.Tier1`–`Tier.Tier5`, business importance) and certification (Bronze/Silver/Gold).
- Classifications and tags: the system `PII` vocabulary, auto vs manual labels, fixing a wrong auto tag, and your own classification.
- Glossary: business terms with synonyms and reviewers, linked to columns — and how that changes search.
- Domains and data products: grouping assets by business area and publishing a bundle for others.

This chapter is Level 2 of the tutorial tree: it deepens the "govern" job from chapter 01 ("who owns it, is it personal data?") and adds meaning to the "understand" job. The question it answers: *how do we say who is responsible, how important and how sensitive each asset is, and what business words mean — in a way people and programs can use?* It assumes Levels 0–2 (the stack is up, Brightcup is cataloged, you can PATCH an entity from chapter 05). Everything below runs against Brightcup through the REST API, from this notebook.

*The cell below is the standard setup: find the project root from any working directory, put its `src/` on the path, and (re)load this chapter's helper module.*

In [1]:
import sys
import importlib
from pathlib import Path

here = Path.cwd()
ROOT = next(d for d in [here, *here.parents, here / "project"] if (d / "pyproject.toml").exists() and (d / "src" / "openmetadata_tutorial").exists())
sys.path.insert(0, str(ROOT / "src"))

from openmetadata_tutorial import nb
ROOT = nb.setup()

from openmetadata_tutorial import ch08_governance as G
G = importlib.reload(G)  # a kernel started earlier holds the old copy otherwise
from openmetadata_tutorial import client

## The whole idea in one picture

Governed, for one asset, means five small facts are written down where people and programs can read them: an **owner** (who is responsible), a **tier** (how important), **classification tags** (how sensitive), a **glossary term** (what it means), and a **domain / data product** (which business area publishes it). In OpenMetadata all five are the same machinery — tags and references on the entity — set by API or UI and versioned like any other change:

```mermaid
flowchart LR
    A[analytics.daily_revenue] --> O[owner: team Finance]
    A --> T[tier: Tier.Tier1]
    A --> C[certification: Gold]
    A --> D[domain: Sales]
    A --> P[data product: Revenue Reporting]
    A --> Gc[revenue column ← glossary Brightcup.Revenue]
```

The running example stays Brightcup. The last section prints one governance report over all seven cataloged assets — these are its rows:

In [2]:
for fqn in G.ALL_ASSETS:
    print(fqn)
print(len(G.ALL_ASSETS), "assets")

brightcup_pg.brightcup.raw.customers
brightcup_pg.brightcup.raw.products
brightcup_pg.brightcup.raw.orders
brightcup_pg.brightcup.raw.order_items
brightcup_pg.brightcup.analytics.fct_orders
brightcup_pg.brightcup.analytics.daily_revenue
brightcup_pg.brightcup.analytics.customer_ltv
7 assets


## 1 — Simple: an owner

An **owner** is the user or team responsible for an asset — the name that shows on the table page and the address alerts and incidents go to. We make owners **teams, not people**: people leave, teams persist. Two teams: `Finance` (owns finance reporting) and `Data Engineering` (owns raw ingestion). Plus `dana.analyst`, an analyst in team Finance — chapter 09 reuses this user, so we do not give it admin.

In [3]:
fin = G.ensure_team("Finance", description="Owns finance reporting")
eng = G.ensure_team("Data Engineering", description="Owns raw ingestion")
dana = G.ensure_user("dana.analyst", "dana.analyst@brightcup.example",
                     teams=["Finance"], description="Brightcup analyst")
print("teams:", fin["name"], fin["_how"], "|", eng["name"], eng["_how"])
print("user:", dana["name"], dana["_how"], "| member of:", dana["_teams"], "| admin:", dana["isAdmin"])

teams: Finance reused | Data Engineering reused
user: dana.analyst reused | member of: ['Finance'] | admin: False


*Now put the teams on the assets: Finance owns the finance number, Data Engineering owns the raw tables it is built from. Owners are set with a JSON Patch on `/owners` (chapter 05 skills) and read back with `fields=owners`.* This stack is shared and already governed, so expect `reused` and `already ...` below — a first run prints `created` and `owner added` instead.

In [4]:
print(G.set_owners(G.DAILY_REVENUE, "Finance"))
print(G.set_owners(G.ORDERS, "Data Engineering"))
print(G.set_owners(G.CUSTOMERS, "Data Engineering"))
print("daily_revenue owners now:", G.owners_of(G.DAILY_REVENUE))

(['Finance'], 'already owner')
(['Data Engineering'], 'already owner')
(['Data Engineering'], 'already owner')


daily_revenue owners now: ['Finance']


`daily_revenue` is owned by team Finance; `raw.orders` and `raw.customers` by Data Engineering. Every helper here is idempotent — run the cell twice and the second run reports "already owner" instead of duplicating anything. The same holds for the whole chapter: `already set`, `already tagged`, `already in domain`, `already certified` all mean the steady state was already in place; a first run prints the doing-words instead (`tier set`, `tag added`, `domain assigned`, `certified`).

> **Important:** Make owners teams, not people, where possible.
> **What it means for us:** `daily_revenue` is owned by Finance, and `dana.analyst` inherits that context by membership — nothing here depends on one person staying.
> **Why it matters:** people leave; alerts and incidents addressed to a team still land (chapter 07's failures become somebody's job, chapter 09 routes them).

## 2 — Tier: how important is it?

A **tier** is the importance tag: `Tier.Tier1` (critical source of truth) down to `Tier.Tier5`. Tiers live in the system `Tier` **classification** — a classification is a vocabulary, a tag is one value in it — and `Tier` is mutually exclusive, so an asset carries at most one. The finance number is Tier1; the raw orders it is built from are Tier2:

In [5]:
print("daily_revenue:", G.set_tier(G.DAILY_REVENUE, "Tier1"))
print("orders:", G.set_tier(G.ORDERS, "Tier2"))

daily_revenue: ('Tier.Tier1', 'already set')
orders: ('Tier.Tier2', 'already set')


*Tiers are searchable facets. This query asks the search index for every table carrying `Tier.Tier1` (`tier.tagFQN`, the same field the UI facet uses):*

In [6]:
fqns, how = G.tier_search("Tier.Tier1")
print(how, "->", fqns)

live -> ['brightcup_pg.brightcup.analytics.daily_revenue']


Only `daily_revenue` comes back. **What it means for us:** tiers drive priority — Tier1 assets get the strictest tests, the loudest alerts and the first review slots. They are a label, not a lock: nothing is enforced until chapter 09 turns labels into policies.

## 3 — Medium: classification, auto and manual

A **classification** is your vocabulary (system ones: `PII`, `Tier`, `Certification`); a **tag** is one value applied to a table or a column (`PII.Sensitive` on `customers.email`). Each applied tag records its provenance: `labelType` (`Generated` = a machine put it there, `Manual` = a human did) and `state` (`Suggested` vs `Confirmed`). The auto-classification workflow (chapter 01/04) already tagged `raw.customers`. Here is what the four interesting columns carry:

In [7]:
for col in ["full_name", "email", "phone", "signup_date"]:
    det = G.column_tag_detail(G.CUSTOMERS, col)
    print(col, "->", [(d["tagFQN"], d["labelType"], d["state"]) for d in det] or "no tags")
    for d in det:
        print("   by", d["appliedBy"], "|", d["reason"])

full_name -> [('PII.Sensitive', 'Generated', 'Suggested')]
   by ingestion-bot | Detected by `SpacyRecognizer` 49 times with an average score of 0.85.
email -> [('PII.Sensitive', 'Manual', 'Confirmed')]
   by ingestion-bot | 
phone -> [('PII.NonSensitive', 'Generated', 'Suggested')]
   by ingestion-bot | Detected by `ItVatCodeRecognizer` 3 times with an average score of 1.00.
signup_date -> no tags


Read it as: `full_name` is `PII.Sensitive` and `phone` is `PII.NonSensitive` — both stamped `Generated`/`Suggested` by `ingestion-bot`, with the `reason` naming the recognizer that fired (`SpacyRecognizer` for names, `ItVatCodeRecognizer` for phones). `email` is the odd one out: it already reads `Manual`/`Confirmed`, human-confirmed by this chapter's earlier run (a re-run keeps it that way — the last call below prints `already confirmed`). The fourth column is this chapter's standing example: on a fresh stack `signup_date` carries `PII.NonSensitive` (`Generated`/`Suggested`) — a false positive, since a signup date identifies nobody. On this already-governed run the earlier correction still stands, so `signup_date` reads `no tags` above. To show the correction with real output, the next cell first puts the classifier's exact label back, then removes it:

In [8]:
print("put the classifier label back:", G.reapply_generated_tag(G.CUSTOMERS, "PII.NonSensitive", column="signup_date"))
print("remove false positive:", G.remove_tag(G.CUSTOMERS, "PII.NonSensitive", column="signup_date"))
print("confirm email:", G.confirm_tag(G.CUSTOMERS, "PII.Sensitive", column="email"))

put the classifier label back: (['PII.NonSensitive'], 'tag added')
remove false positive: ([], 'tag removed')
confirm email: ([('PII.Sensitive', 'Manual', 'Confirmed')], 'already confirmed')


*Classifications are also yours to invent. Brightcup's tables are synthetic test data, and that fact should travel with them — a custom `DataSource` vocabulary with `Synthetic` and `Production`, tagged on the two raw tables the tutorial tests (and only those):*

In [9]:
print(G.ensure_classification("DataSource", "Where the data came from")["fullyQualifiedName"])
print(G.ensure_tag("DataSource", "Synthetic", "Generated test data")["fullyQualifiedName"])
print(G.ensure_tag("DataSource", "Production", "Real shop data")["fullyQualifiedName"])
print("orders:", G.add_tag(G.ORDERS, "DataSource.Synthetic"))
print("customers:", G.add_tag(G.CUSTOMERS, "DataSource.Synthetic"))

DataSource
DataSource.Synthetic
DataSource.Production
orders: (['DataSource.Synthetic', 'Tier.Tier2'], 'already tagged')
customers: (['DataSource.Synthetic', 'PII.Sensitive'], 'already tagged')


> **Important:** Auto classification suggests; a human confirms.
> **What it means for us:** `signup_date` carries no PII tag anymore and `email` is `Manual`/`Confirmed` — the catalog now says what a person checked, not just what a recognizer guessed. That correction stands only until the next classify run: re-running `just ingest classify classify` re-adds removed `Generated` tags (the shared `ingestion/classify.yaml` filters whole schemas, not columns, so there is nowhere to exempt `signup_date` — stop the schedule or re-run this section's removal after every classify; the chapter test re-runs it too).
> **Why it matters:** wrong PII tags lead to wrong access decisions — chapter 09's policies read these very tags, so a false positive can lock out legitimate readers (or a false negative can expose emails).

## 4 — Glossary: what do the words mean?

A **glossary** is the business vocabulary: terms with definitions, synonyms and relations to each other. Four terms for the `Brightcup` glossary — note `Revenue`'s synonyms (`sales`, `turnover`) and the relations (`Customer`↔`Order`, `Lifetime value`↔`Revenue`):

In [10]:
print(G.ensure_glossary("Brightcup", "Brightcup business vocabulary")["fullyQualifiedName"])
terms = [
    ("Revenue", "Sum of total_amount of non-cancelled orders, in EUR, by order date.", ["sales", "turnover"], []),
    ("Order", "One checkout by a customer; may contain several order items.", [], []),
    ("Customer", "A person with an account in the Brightcup web shop.", [], ["Brightcup.Order"]),
    ("Lifetime value", "Total revenue from one customer since signup.", [], ["Brightcup.Revenue"]),
]
for name, desc, syns, rel in terms:
    t = G.ensure_term("Brightcup", name, description=desc, synonyms=syns, related=rel)
    print(t["fullyQualifiedName"], "|", t.get("entityStatus"), "|", t["_how"])

Brightcup
Brightcup.Revenue | Approved | reused
Brightcup.Order | Approved | reused
Brightcup.Customer | Approved | reused
Brightcup.Lifetime value | Approved | reused


*A glossary nobody links is a wiki nobody reads. Link the meaning to the data: `Revenue` onto the `revenue` column of `daily_revenue`, `Lifetime value` onto `lifetime_value` of `customer_ltv`. Technically a link is a tag with `source: Glossary`:*

In [11]:
print("revenue column:", G.link_term_to_column("Brightcup.Revenue", G.DAILY_REVENUE, "revenue"))
print("lifetime_value column:", G.link_term_to_column("Brightcup.Lifetime value", G.ANALYTICS + ".customer_ltv", "lifetime_value"))

revenue column: (['Brightcup.Revenue'], 'already tagged')
lifetime_value column: (['Brightcup.Lifetime value'], 'already tagged')


*Do the synonyms help search? Ask the index — honestly, on this stack, with this result:*

In [12]:
hit = client.get("/v1/search/query", q="turnover", index="table_search_index", size=5)
print("q=turnover on tables:", hit["hits"]["total"], [h["_source"].get("fullyQualifiedName") for h in hit["hits"]["hits"]])
g = client.get("/v1/search/query", q="Revenue", index="glossary_term_search_index", size=5)
names = [h["_source"].get("fullyQualifiedName") for h in g["hits"]["hits"]]
print("q=Revenue on glossary terms:", names)
for n in names:
    print("-", n, "|", client.get(f"/v1/glossaryTerms/name/{n}").get("description"))

q=turnover on tables: {'relation': 'eq', 'value': 0} []
q=Revenue on glossary terms: ['Brightcup.Revenue', 'Brightcup.Lifetime value']
- Brightcup.Revenue | Sum of total_amount of non-cancelled orders, in EUR, by order date.
- Brightcup.Lifetime value | Total revenue from one customer since signup.


**What it means for us:** in 2.0.5 the synonym does *not* pull `daily_revenue` into a `turnover` table search (zero hits, observed) — but the terms are found, and the descriptions printed above show why each matches `Revenue`: `Revenue` by name, `Lifetime value` by its description (`Total revenue from one customer since signup.`). The column page shows the meaning where it counts. Do not promise synonym search in this version; promise linked context, which also feeds chapter 09's AI assistant.

*Terms with reviewers get a workflow.* Ask `admin` to review `Order`: the term moves to In Review (the move is async — a background workflow flips the status seconds after reviewers are set, so we poll), and while it is there the ingestion bot can no longer edit it:

In [13]:
print("reset reviewers:", G.reset_reviewers("Brightcup.Order")["fullyQualifiedName"])
print("ask admin to review:", G.set_reviewers("Brightcup.Order", ["admin"])["fullyQualifiedName"])
print("status after the workflow reacts:", G.wait_status("Brightcup.Order")["entityStatus"])

reset reviewers: Brightcup.Order
ask admin to review: Brightcup.Order


status after the workflow reacts: In Review


In [14]:
print("bot edit attempt:", G.bot_update_attempt("Brightcup.Order", "bot tries to rewrite this"))
print("admin approves:", G.approve_term("Brightcup.Order").get("entityStatus"))

bot edit attempt: (403, "User 'ingestion-bot' is not a reviewer")


admin approves: Approved


> **Important:** A glossary is only useful when linked to assets and reviewed.
> **What it means for us:** `Revenue` sits on the `revenue` column (not in a side document), and `Order` walks In Review → Approved with an enforced reviewer — cell 29 polls the async move to In Review, cell 30 shows the bot refused with 403 and the admin approval returning Approved. Section 4's term rows are the idempotent steady state (`reused` on every run): each reads Approved when no review is open, In Review while one is — check the four rows above against that rule.
> **Why it matters:** an unlinked glossary is a wiki nobody reads; linked, reviewed terms are the context AI assistants quote (chapter 09) instead of guessing what "revenue" means.

## 5 — Domains and data products: who publishes what?

A **domain** is a business area (`Sales`); a **data product** is a curated bundle inside it (`Revenue Reporting`) — what one team publishes for others to consume. Create the `Sales` domain (type Consumer-aligned, owned by Finance) and assign the five Brightcup assets that belong to sales:

In [15]:
print(G.ensure_domain("Sales", "Brightcup sales business area", owner_team="Finance")["fullyQualifiedName"])
for fqn in [G.ORDERS, G.CUSTOMERS, G.ANALYTICS + ".fct_orders", G.DAILY_REVENUE, G.ANALYTICS + ".customer_ltv"]:
    print(fqn.split(".")[-1], G.set_domain(fqn, "Sales"))

Sales
orders (['Sales'], 'already in domain')
customers (['Sales'], 'already in domain')
fct_orders (['Sales'], 'already in domain')
daily_revenue (['Sales'], 'already in domain')
customer_ltv (['Sales'], 'already in domain')


*Then the data product. It bundles `daily_revenue` — plus the `Finance overview` dashboard from chapter 06 if it exists. On this stack no dashboard exists yet, so the helper skips it openly (`_skipped`) and the product holds a single asset — the honest state, not the full spec shape. Four read-backs follow: the linked asset FQNs, the table-side GET (the link that persists in 2.0.5), the product-side GET (which reads back `null` — that field is deprecated, observed), and the search index:*

In [16]:
dp = G.ensure_data_product("Revenue Reporting", "Sales", "Curated revenue assets for finance",
                           assets=[("table", G.DAILY_REVENUE), ("dashboard", "Finance overview")])
print("linked assets:", dp.get("_assets"))
print("skipped (absent on this stack):", dp.get("_skipped"))
print("table-side GET:", G.data_products_of(G.DAILY_REVENUE))
print("product-side GET fields=assets:", client.get("/v1/dataProducts/name/Revenue Reporting", fields="assets").get("assets"))
s = client.get("/v1/search/query", q="Revenue Reporting", index="data_product_search_index", size=1)
doc = s["hits"]["hits"][0]["_source"]
print("search finds:", doc["fullyQualifiedName"])
print("search doc domains:", [d["name"] for d in doc.get("domains", [])],
      "| owners:", [o["name"] for o in doc.get("owners", [])],
      "| assets:", doc.get("assets"))

linked assets: ['brightcup_pg.brightcup.analytics.daily_revenue']
skipped (absent on this stack): ['Finance overview']


table-side GET: ['Revenue Reporting']
product-side GET fields=assets: None
search finds: Revenue Reporting
search doc domains: ['Sales'] | owners: ['Finance'] | assets: None


> **Important:** Domains change what people see.
> **What it means for us:** the five assets above are now in `Sales` — selecting "Sales" in the UI domain switch (chapter 02) filters Explore to them; assets outside stay hidden behind that filter. `daily_revenue` is additionally published in the `Revenue Reporting` data product: the linked FQNs above are the membership list, and the table-side GET, the search index and the report all show it.
> **Why it matters:** assets "disappear" for users who selected another domain, and domain-based policies (chapter 09) decide who may edit them. One 2.0.5 limit, observed above: the product-side `assets` field is deprecated and reads back `null` — the table-side `/dataProducts` back-reference is the link that persists.

## 6 — Certification: is it ready to use?

**Certification** stamps readiness: Bronze, Silver, Gold — a tag in the system `Certification` classification, set through `/certification`. The finance number is Gold:

In [17]:
print("certify:", G.set_certification(G.DAILY_REVENUE, "Gold"))
hit = client.get("/v1/search/query", q="daily_revenue", index="table_search_index", size=1)
src = hit["hits"]["hits"][0]["_source"]
print("tier in search:", src.get("tier", {}).get("tagFQN"))
print("certification in search:", (src.get("certification") or {}).get("tagLabel", {}).get("tagFQN"))

certify: ('Certification.Gold', 'already certified')
tier in search: Tier.Tier1
certification in search: Certification.Gold


**What it means for us:** search now returns `daily_revenue` with `Tier.Tier1` and `Certification.Gold` attached — a consumer (or an AI agent) can see importance *and* readiness without opening the page.

## 7 — Complex: one governance report

Assembly: one row per asset — owners, tier, certification, domain, data products, PII columns, glossary terms. This is what "governed" means concretely:

In [18]:
import pandas as pd
pd.set_option("display.max_colwidth", None)
df = G.report()
print(df.to_string())
print(len(df), "assets")
print("customers PII in full:", df.set_index("fqn").loc[G.CUSTOMERS, "PII columns"])

                                              fqn            owners        tier certification domain      data products                                                                                PII columns                 glossary terms
0            brightcup_pg.brightcup.raw.customers  Data Engineering                            Sales                     full_name (Sensitive/Generated), email (Sensitive/Manual), phone (NonSensitive/Generated)                               
1             brightcup_pg.brightcup.raw.products                                                                                                                                  origin (NonSensitive/Generated)                               
2               brightcup_pg.brightcup.raw.orders  Data Engineering  Tier.Tier2                Sales                                                                             order_ts (NonSensitive/Generated)                               
3          brightcup_pg.brightcu

In [19]:
cfg = G.metadata_source_config()
print("overrideMetadata flag in ingestion/metadata.yaml:", cfg["override_flag_present"])
print(cfg["source_config"])

overrideMetadata flag in ingestion/metadata.yaml: False
config:
      type: DatabaseMetadata
      includeTables: true
      includeViews: true
      markDeletedTables: true
      schemaFilterPattern:
        includes:
          - ^raw$
          - ^analytics$



Read it row by row. `daily_revenue` is the model citizen: owner Finance, Tier1, Gold, domain Sales, product Revenue Reporting, no PII, meaning linked. The empty cells are the to-do list, not a failure — `products` has no owner and no domain, `fct_orders` and `customer_ltv` have no tier yet. (Anything you did not set yourself, like extra tags on `products` or `order_items`, is other chapters' work on this shared stack — the report makes everyone's state visible, which is rather the point.)

*Will the next ingestion wipe this?* For the metadata workflow: no — and instead of a re-run claim, here is the evidence. The workflow's `sourceConfig` (printed above) sets no `overrideMetadata` flag, i.e. the toggle stays off (the default), and per the connector docs, "when off, OpenMetadata only updates fields that have no existing value" (docs.open-metadata.org, v2.0.x connector docs) — so owners, tiers, domains, certification and glossary links set here survive metadata ingestion. Two exceptions, both visible in this chapter: re-running the *classify* workflow re-adds removed `Generated` PII tags (section 3's removal is a post-classify step), and flipping the override toggle on would let source values overwrite descriptions, tags, owners and display names. So: govern freely, but check that toggle before scheduling anything new.

The governed table page is saved at `assets/08_daily_revenue_governed.png`, taken after this notebook ran (`uv run python -m openmetadata_tutorial.ch08_shot` from `project/`) — owner, tier, certification, domain, data product and the glossary term (`Revenue` on the `revenue` row) on one page. It is deliberately not `just shot`: the columns table is wider than its card, so the Glossary Terms column needs a sideways scroll the standard capture cannot do — the chapter helper repeats the same login, scrolls the table right, and shoots.

## Carry forward

- Governance = **owner + tier + classification + meaning (glossary) + grouping (domain/data product)** on each asset — five small facts, all set above on Brightcup.
- Everything is **tags and references on entities**, set by API or UI and versioned like any change — the same PATCH shapes from chapter 05.
- The governance report's **empty cells are the backlog** — the next governed asset is whichever row you fill in next (chapter 09 turns these labels into access rules, alerts and promises).